Data exploration of sunrgbd

In [1]:
#Imports
import os
import numpy as np
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image
import torch
from torch.utils.data import Dataset, DataLoader, Subset
from sklearn.model_selection import KFold

from data_handling import SUNRGBDDataset1
from data_handling import SUNRGBDDataset2

In [2]:
#load dataset
SUNRGBD_ROOT = './'

num_classes = 13
dataset = SUNRGBDDataset2(
    root=SUNRGBD_ROOT,
    num_classes=num_classes,
    transform=None,
    img_size=(480, 640)
)

Found 10335 samples. Images will be read from disk on demand.
Scanning mask files for unique classes (reads label PNGs only)...
  Scanning 0/10335...
  Scanning 500/10335...
  Scanning 1000/10335...
  Scanning 1500/10335...
  Scanning 2000/10335...
  Scanning 2500/10335...
  Scanning 3000/10335...
  Scanning 3500/10335...
  Scanning 4000/10335...
  Scanning 4500/10335...
  Scanning 5000/10335...
  Scanning 5500/10335...
  Scanning 6000/10335...
  Scanning 6500/10335...
  Scanning 7000/10335...
  Scanning 7500/10335...
  Scanning 8000/10335...
  Scanning 8500/10335...
  Scanning 9000/10335...
  Scanning 9500/10335...
  Scanning 10000/10335...
Classes present in dataset: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13]


Image properties:

In [57]:
#amount of images
print("Amount of images: ", len(dataset))

#Size of images
print("Image dimensions: ", dataset.__getitem__(0)[2].shape)

#Amount of classes. Predefined above using the https://github.com/chrischoy/SUN_RGBD/tree/master
print("Amount of classes: ", num_classes)

#What does each class correspond to?
class_dict = {
    0:"Other",
    1:"Bed",
    2:"Books",
    3:"Ceiling",
    4:"Chair",
    5:"Floor",
    6:"Furniture",
    7:"Objects",
    8:"Picture",
    9:"Sofa",
    10:"Table",
    11:"TV",
    12:"Wall",
    13:"Window"
}
print("All classes:")
for key, value in class_dict.items():
    print(f"{key} \t= {value}")

#Are all images the same size?
image_same_shape = True
missing_values = False
img_shape = dataset[0][0].shape
for (rgb,depth,mask) in dataset:
    image = rgb
    if img_shape != image.shape:
        image_same_shape = False
    if np.isnan(image.numpy()).any():
        missing_values = True
    if image_same_shape == False and missing_values == True:
        break
print("Do images have the same shape: ", image_same_shape)

#Any missing values?
print("Are there missing values: ", missing_values)

Amount of images:  10335
Image dimensions:  torch.Size([480, 640])
Amount of classes:  13
All classes:
0 	= Other
1 	= Bed
2 	= Books
3 	= Ceiling
4 	= Chair
5 	= Floor
6 	= Furniture
7 	= Objects
8 	= Picture
9 	= Sofa
10 	= Table
11 	= TV
12 	= Wall
13 	= Window
Do images have the same shape:  True
Are there missing values:  False


Counting block

In [58]:
#counting how many of each class out of total pixels
counter = [(key,0) for key, value in class_dict.items()]
unique_image_counter = [(key,0) for key, value in class_dict.items()]
for idx, (rgb, depth, mask) in enumerate(dataset):
    if idx % 500 == 0:
        print(f"Finished {idx}/{len(dataset)}")
    mask_np = mask.numpy()
    unique, counts = np.unique(mask_np, return_counts=True)
    for (key, count) in zip(unique,counts):
        counter[key] = (counter[key][0],int(count+counter[key][1]))
        
        if count > 0:
            exist_in_image = 1
        else:
            exist_in_image = 0
        unique_image_counter[key] = (unique_image_counter[key][0],int(unique_image_counter[key][1] + exist_in_image))
height, width = dataset.__getitem__(0)[2].shape
total_pixels = int(height) * int(width) * len(dataset)
percentage_counter = [(key, value/total_pixels) for (key, value) in counter]
unique_image_percentage_counter = [(key, value/len(dataset)) for (key, value) in unique_image_counter]

Finished 0/10335
Finished 500/10335
Finished 1000/10335
Finished 1500/10335
Finished 2000/10335
Finished 2500/10335
Finished 3000/10335
Finished 3500/10335
Finished 4000/10335
Finished 4500/10335
Finished 5000/10335
Finished 5500/10335
Finished 6000/10335
Finished 6500/10335
Finished 7000/10335
Finished 7500/10335
Finished 8000/10335
Finished 8500/10335
Finished 9000/10335
Finished 9500/10335
Finished 10000/10335


Pixel statistics for masks

In [59]:
#total print
print("Percentage of total pixels for each class")
for (key, count) in counter:
    print(f"{class_dict[key]} = {count}")

#percentage print
print("\nPercentage of total pixels for each class")
for (key, percentage) in percentage_counter:
    print(f"{class_dict[key]} = {percentage*100:.2f}%")

Percentage of total pixels for each class
Other = 805543805
Bed = 82311696
Books = 11464710
Ceiling = 25542497
Chair = 258206247
Floor = 536066995
Furniture = 149237598
Objects = 129116277
Picture = 18297460
Sofa = 71232960
Table = 254566958
TV = 5019562
Wall = 717578131
Window = 110727104

Percentage of total pixels for each class
Other = 25.37%
Bed = 2.59%
Books = 0.36%
Ceiling = 0.80%
Chair = 8.13%
Floor = 16.88%
Furniture = 4.70%
Objects = 4.07%
Picture = 0.58%
Sofa = 2.24%
Table = 8.02%
TV = 0.16%
Wall = 22.60%
Window = 3.49%


Class statistics per image

In [61]:
#How many images are each class in?
print("\nTotal amount of images each class is in")
for (key, count) in unique_image_counter:
    #print("Key:", key)
    #print("count:",count)
    print(f"{class_dict[key]} = {count}")

#percentage of total
print("\nPercentage of images this class is in")
for (key, percentage) in unique_image_percentage_counter:
    print(f"{class_dict[key]} = {percentage*100:.2f}%")


Total amount of images each class is in
Other = 10333
Bed = 1208
Books = 1217
Ceiling = 1346
Chair = 5964
Floor = 8945
Furniture = 3204
Objects = 5904
Picture = 1755
Sofa = 1310
Table = 6272
TV = 296
Wall = 9640
Window = 3652

Percentage of images this class is in
Other = 99.98%
Bed = 11.69%
Books = 11.78%
Ceiling = 13.02%
Chair = 57.71%
Floor = 86.55%
Furniture = 31.00%
Objects = 57.13%
Picture = 16.98%
Sofa = 12.68%
Table = 60.69%
TV = 2.86%
Wall = 93.28%
Window = 35.34%
